In [15]:
from dotenv import load_dotenv
load_dotenv()

import dspy

In [16]:
class MySignature(dspy.Signature):
    """Answer the questions using the given context"""
    context: str = dspy.InputField()
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

In [21]:
# Configure Groq LLM
lm = dspy.LM('groq/llama-3.3-70b-versatile')
dspy.configure(lm=lm)

qa = dspy.Predict(MySignature)
response = qa(context="Epstein was a pedophile", question="Who was Epstein?")
print(response.answer)

Epstein was a pedophile, which suggests that he was an individual who had a sexual preference for prepubescent or pubescent children, and his actions were likely those of a child sex abuser.


In [22]:
class Naya(dspy.Signature):
    context : str = dspy.InputField()
    question : str = dspy.InputField()
    choices : list[str] = dspy.InputField()
    best_answer : str = dspy.OutputField()

In [23]:
rerank = dspy.Predict(Naya)
response = rerank(
    context="The Nile is the longest river in Africa.",
    question="What is the longest river in Africa?",
    choices=["Amazon", "Yangtze", "Nile"]
)
print(response.best_answer)

Nile


In [24]:
class COT(dspy.Signature):
    question: str = dspy.InputField()
    reasoning: str = dspy.OutputField(desc="step-by-step reasoning")
    answer: str = dspy.OutputField()

In [25]:
cot = dspy.ChainOfThought(COT)
response = cot(question = "how to implement a 1-bit quantisation using qlora technique to finetune llms?")
print(response.reasoning)
print(response.answer)

To implement 1-bit quantization using the QLoRA (Quantized Low-Rank Adaptation) technique for fine-tuning large language models (LLMs), we need to follow a series of steps. QLoRA is an efficient method for adapting pre-trained language models to specific tasks with minimal additional parameters and computation. The core idea behind QLoRA is to update only the low-rank matrices in the transformer layers, which allows for significant compression and acceleration. When we further apply 1-bit quantization, we are essentially reducing the precision of the model's weights to just 1 bit, which can lead to extreme model compression but requires careful handling to maintain model performance.

1. **Understanding QLoRA**: First, grasp how QLoRA works. It involves adding low-rank matrices to the weight matrices of the linear layers within a transformer model. These low-rank matrices are learned during fine-tuning and are typically much smaller than the original weight matrices, making the adaptat

In [30]:
class QAPipeline(dspy.Module):
    def __init__(self):
        super().__init__()
        self.qa = dspy.Predict(MySignature)

    def forward(self, context, question):
        return self.qa(context=context, question=question)


In [31]:
pipeline = QAPipeline()

res = pipeline(
    context="Paris is the capital of France.",
    question="What is the capital of France?"
)

print(res.answer)


Paris


In [35]:
from dspy.teleprompt import BootstrapFewShot

# Training data should be dspy.Example objects
training_data = [
    dspy.Example(
        context="Einstein was a physicist.",
        question="Who was Einstein?",
        answer="Einstein was a renowned physicist."
    ).with_inputs("context", "question"),
    dspy.Example(
        context="Newton discovered gravity.",
        question="Who discovered gravity?",
        answer="Newton discovered gravity."
    ).with_inputs("context", "question"),
]

# Create the pipeline
pipeline = dspy.Predict(MySignature)

# Compile with BootstrapFewShot (no valset parameter)
comp = BootstrapFewShot()
optimized_pipeline = comp.compile(pipeline, trainset=training_data)

# Test it
result = optimized_pipeline(
    context="The Sun is a star.",
    question="What is the Sun?"
)
print(result.answer)

100%|██████████| 2/2 [00:00<00:00,  3.27it/s]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
The Sun is a star.
